## Setup Test

In [1]:
import duckdb

con = duckdb.connect()  # in-memory, no server needed

# Load all CSVs — note the relative path goes UP one level from notebooks/
for table in ['users', 'tracks', 'artists', 'sessions', 'events']:
    con.execute(f"""
        CREATE TABLE {table} AS
        SELECT * FROM read_csv_auto('../daily_mix_data/{table}.csv')
    """)
    count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table}: {count:,} rows loaded")

print("\n--- D30 Retention by Variant ---")
result = con.execute("""
    WITH d30 AS (
        SELECT DISTINCT e.user_id
        FROM events e
        JOIN users u ON e.user_id = u.user_id
        WHERE CAST(e.event_time AS DATE) = CAST(u.signup_date AS DATE) + 30
          AND e.event_type IN ('play', 'app_open', 'daily_mix_play')
    )
    SELECT
        u.experiment_variant,
        COUNT(DISTINCT u.user_id) AS cohort,
        COUNT(DISTINCT d30.user_id) AS retained,
        ROUND(100.0 * COUNT(DISTINCT d30.user_id)
            / COUNT(DISTINCT u.user_id), 2) AS d30_pct
    FROM users u
    LEFT JOIN d30 ON u.user_id = d30.user_id
    GROUP BY u.experiment_variant
""").fetchdf()

print(result.to_string(index=False))

  users: 12,000 rows loaded
  tracks: 2,441 rows loaded
  artists: 500 rows loaded
  sessions: 97,945 rows loaded
  events: 747,848 rows loaded

--- D30 Retention by Variant ---
experiment_variant  cohort  retained  d30_pct
         daily_mix    5935      1700    28.64
           control    6065      1414    23.31
